# Laboratory 18 — Fluctuations and transport

In this laboratory the noise the course has spent seventeen modules averaging away goes to work.
You read a heat capacity off the energy fluctuations of a system on a bath, with nothing heated;
watch the relative fluctuation fall as $N^{-1/2}$; release ten thousand random walkers and watch
them melt into the diffusion equation's solution; measure their diffusion coefficient three
independent ways; give them a memory of their velocity and watch the simplest formula for $D$
fail while the integral of the velocity memory does not; watch module 04's gas fail to diffuse at
all; estimate the mean free path and the transport coefficients of real gases and test them
against measurements; and finally weigh an atom, as Perrin did, from the Brownian motion of a
grain of resin.

The route: the objects (Part 1); a heat capacity from the noise (Part 2); a longer run (Part 3);
the $N^{-1/2}$ law (Part 4); the walker cloud (Part 5); $D$ three ways (Part 6); walkers with memory
(Part 7); module 04's box, which cannot diffuse (Part 8); collisions in a real gas (Part 9);
viscosity and conductivity against measurement (Part 10); Perrin's atoms (Part 11); a diffusion
experiment of your own (Part 12); the automated checks (Part 13); and four explorers (Part 14).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

**Units.** SI throughout. Walkers live on a line and are measured in their own units: lengths in
steps $\ell$ and times in step times $\tau$, so $D = \ell^2/2\tau$ is $0.5$. The gases of Parts 9 and
10 are three-dimensional and in metres and seconds.

## Model specification

| | |
|---|---|
| **System** | independent random walkers on a line, standing in for tagged molecules; and, for the fluctuations, module 04's gas and module 11's two-level systems in contact with a bath |
| **Dynamics** | steps of $\pm\ell$ every $\tau$, independent, or repeating the last step with probability $q$ and otherwise drawn afresh; the estimators use only the recorded positions and velocities |
| **Boundary** | an unbounded line for the walkers; module 04's rigid box in Part 8; a heat bath at $T$ for the fluctuations |
| **Ensemble** | independent walkers realise the ensemble directly; the energies are canonical draws, each independent of the last |
| **Ignored** | walker–walker interactions, convection and drift, any memory beyond one-step persistence |
| **Valid when** | times long compared with the velocity memory; many walkers; steps of finite variance |
| **Failure modes** | correlated steps ($D = \ell^2/2\tau$ fails); times comparable with the memory (ballistic); drift or convection; walls (the spread saturates) |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thermolab import gases, kinetics, partition, sampling, transport
from thermolab.constants import AMU, K_B, N_A
from thermolab.validation import scaling_exponent

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

M_AR = 39.948 * AMU   # an argon atom [kg]
GAP = 1e-21           # the two-level systems' gap [J]
STEP, TAU = 1.0, 1.0  # a walker's step and step time: walkers are measured in their own units


def read_table(name):
    # A data file as a dict of columns; every line starting with # is commentary.
    lines = [line for line in (DATA / name).read_text(encoding="utf-8").splitlines()
             if line and not line.startswith("#")]
    header = lines[0].split(",")
    columns = zip(*(line.split(",") for line in lines[1:]), strict=True)
    return {h: np.array(c) if h == "gas" else np.array(c, dtype=float)
            for h, c in zip(header, columns, strict=True)}


print(f"two-level gap / k_B = {GAP / K_B:.1f} K; walker D = l^2/(2 tau) = "
      f"{transport.persistent_walk_d(STEP, TAU)} l^2/tau")

## Part 1 — The objects

| function | what it gives |
|---|---|
| `gas_energy_samples(N, T, m, n, rng, ...)`, `two_level_energy_samples(N, gap, T, n, rng)` | `n` independent canonical energies of module 04's gas or of $N$ two-level systems |
| `energy_fluctuation_cv(energies, T)` | $C_V = \sigma_E^2/\kB T^2$ with its error bar |
| `walker_trajectories(n, steps, l, rng, persistence)`, `walker_velocities` | walkers' positions at every step, and their velocities |
| `msd`, `diffusion_greens_function`, `step_profile` | the mean squared displacement; the point-source and dye-layer solutions of the diffusion equation |
| `d_from_msd`, `d_from_histogram`, `d_from_vacf` | three estimates of $D$, each an `Estimate` with `.value` and `.error` |
| `velocity_autocorrelation`, `green_kubo_running`, `persistent_walk_d` | $C(k)$; its running integral; the exact $D$ of a walk with persistence $q$ |
| `box_gas_trace` | module 04's gas, recorded step by step |
| `kinetics.mean_free_path`, `kinetics.collision_rate`, `kinetic_estimates` | $\lambda$, $\nu$ and the trio $D$, $\eta$, $\kappa$ of a dilute hard-sphere gas |
| `stokes_friction`, `einstein_diffusion`, `avogadro_from_diffusion` | Stokes drag, the Einstein relation, and Perrin's $N_A$ |

Before you run anything, write down your answers to the four questions on the module page.

**Your predictions:**

1.
2.
3.
4.

## Part 2 — A heat capacity from the noise

Draw module 04's gas many times over, each time with fresh Maxwell–Boltzmann velocities and no
rescaling (`fix_temperature=False`), so that its energy fluctuates from copy to copy as a canonical
system's does. Read $C_V = \sigma_E^2/\kB T^2$ off the scatter. Then do the same for $1000$ two-level
systems at four temperatures, and compare with module 12's closed form and with a heat capacity
measured the ordinary way, by changing the temperature by $\pm 5\%$ and watching the mean energy.

**Your prediction:** what will the noise say $C_V$ is for $100$ argon atoms, in units of $N\kB$? And
for the same gas drawn microcanonically, with its energy fixed?

*(write here before running the next cell)*

In [ ]:
rng = np.random.default_rng(1801)
T = 300.0
argon = transport.energy_fluctuation_cv(transport.gas_energy_samples(100, T, M_AR, 10_000, rng), T)
print(f"100 argon atoms, canonical: C_V / (N k_B) = {argon.value / (100 * K_B):.3f} "
      f"+- {argon.error / (100 * K_B):.3f}   (equipartition: 1.5)")
four = {drift: transport.energy_fluctuation_cv(
    transport.gas_energy_samples(4, T, M_AR, 20_000, rng, remove_drift=drift), T)
    for drift in (False, True)}
print(f"4 argon atoms: C_V / (N k_B) = {four[False].value / (4 * K_B):.3f} +- "
      f"{four[False].error / (4 * K_B):.3f}; with each copy's net momentum removed "
      f"{four[True].value / (4 * K_B):.3f} +- {four[True].error / (4 * K_B):.3f}   "
      f"((3/2)(N - 1)/N = 1.125)")
fixed = transport.energy_fluctuation_cv(
    transport.gas_energy_samples(100, T, M_AR, 500, rng, fix_temperature=True), T)
print(f"100 argon atoms, microcanonical: C_V / (N k_B) = {fixed.value / (100 * K_B):.1e}")

print(f"\n1000 two-level systems, gap / k_B = {GAP / K_B:.1f} K; heat capacity / (N k_B)")
print("  k_B T / gap    from the noise       by heating (+-5%)    exact (module 12)")
two_level = {}
for ratio in (0.25, 0.42, 1.0, 3.0):
    t = ratio * GAP / K_B
    noise = transport.energy_fluctuation_cv(
        transport.two_level_energy_samples(1000, GAP, t, 5000, rng), t)
    hot = transport.two_level_energy_samples(1000, GAP, 1.05 * t, 5000, rng)
    cold = transport.two_level_energy_samples(1000, GAP, 0.95 * t, 5000, rng)
    heated = (hot.mean() - cold.mean()) / (0.1 * t)
    heated_error = np.sqrt((hot.var(ddof=1) + cold.var(ddof=1)) / 5000) / (0.1 * t)
    exact = 1000 * partition.two_level_heat_capacity(GAP, t)
    two_level[ratio] = (noise, heated, heated_error, exact)
    scale = 1000 * K_B
    print(f"  {ratio:6.2f}     {noise.value / scale:.4f} +- {noise.error / scale:.4f}    "
          f"{heated / scale:.4f} +- {heated_error / scale:.4f}     {exact / scale:.4f}")

The noise is a calorimeter. For the gas it reads equipartition's $\tfrac32$ per atom; for four atoms
with their net momentum removed it reads $\tfrac32(N-1)/N$ per atom — the three degrees of freedom frozen by
removing the drift no longer fluctuate, and the noise counts them out. For the two-level systems
it follows module 12's Schottky curve at every temperature, as the ordinary measurement does, but
from data taken at a single temperature. Drawn microcanonically, the gas has no energy fluctuation
at all, and the instrument reads zero: the identity belongs to the system *on a bath*.

## Part 3 — A longer run

If the fluctuation were numerical error, more samples would shrink it. Watch what more samples
actually do.

**Your prediction:** from $100$ to $100\,000$ samples, what happens to $\sigma_E$, and what to the
uncertainty of the mean energy?

*(write here before running the next cell)*

In [ ]:
rng = np.random.default_rng(1802)
t = 0.42 * GAP / K_B
exact_c = 1000 * partition.two_level_heat_capacity(GAP, t)
exact_sigma = np.sqrt(K_B * t**2 * exact_c)
print("  samples    sigma_E / gap    error of <E> / gap    C_V from the noise / exact")
longer = {}
for n in (100, 1000, 10_000, 100_000):
    e = transport.two_level_energy_samples(1000, GAP, t, n, rng)
    c = transport.energy_fluctuation_cv(e, t)
    longer[n] = (e.std(ddof=1), e.std(ddof=1) / np.sqrt(n), c)
    print(f"  {n:7d}      {longer[n][0] / GAP:7.3f}          {longer[n][1] / GAP:7.4f}            "
          f"{c.value / exact_c:.4f} +- {c.error / exact_c:.4f}")
print(f"sqrt(k_B T^2 C_V) / gap = {exact_sigma / GAP:.3f}")

A thousandfold longer run left $\sigma_E$ where it was — at $\sqrt{\kB T^2 C_V}$ — and shrank only the
uncertainty of what was measured: the mean energy's error bar, and the heat capacity's. The
fluctuation is a property of the system, not of the record.

## Part 4 — Why large systems look steady

The relative fluctuation $\sigma_E/\langle E\rangle$ of $N$ two-level systems at $\kB T = \Delta$, for
$N$ from $16$ to $4096$.

**Your prediction:** the slope of $\log(\sigma_E/\langle E\rangle)$ against $\log N$.

*(write here before running the next cell)*

In [ ]:
rng = np.random.default_rng(1803)
t = GAP / K_B
sizes = 2 ** np.arange(4, 13)
relative = []
for n in sizes:
    e = transport.two_level_energy_samples(int(n), GAP, t, 4000, rng)
    relative.append(e.std(ddof=1) / e.mean())
relative = np.array(relative)
exponent = scaling_exponent(sizes, relative)
p_excited = partition.two_level_energy(GAP, t) / GAP

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.loglog(sizes, relative, "o", color="#2563eb", label="measured")
ax.loglog(sizes, np.sqrt((1 - p_excited) / (sizes * p_excited)), "k--",
          label="sqrt(k_B T^2 C_V) / U")
ax.set_xlabel("N")
ax.set_ylabel("sigma_E / <E>")
ax.legend()
plt.tight_layout()
plt.show()
print(f"fitted exponent {exponent:.3f}; sigma_E / <E> = {relative[0]:.3f} at N = 16, "
      f"{relative[-1]:.4f} at N = 4096")
print(f"for a mole of them: {np.sqrt((1 - p_excited) / (N_A * p_excited)):.1e}")

The exponent is $-\tfrac12$, as step 2 derived from the identity, and for a mole the energy is sharp
to two parts in $10^{12}$: the canonical and microcanonical descriptions cannot be told apart.

## Part 5 — The walker cloud

Ten thousand independent walkers, each stepping $\pm\ell$ every $\tau$, all released at the origin.
Their histograms at three times are drawn with the diffusion equation's Green's function for
$D = \ell^2/2\tau$ over them — computed, not fitted — and their mean squared displacement on
logarithmic axes.

**Your prediction:** how much wider is the cloud at $1000\tau$ than at $250\tau$?

*(write here before running the next cell)*

In [ ]:
rng = np.random.default_rng(1804)
walkers = transport.walker_trajectories(10_000, 1000, STEP, rng)
D_EXACT = transport.persistent_walk_d(STEP, TAU)
spread = transport.msd(walkers)
lags = np.unique(np.geomspace(10, 1000, 25).astype(int))
cloud_slope = scaling_exponent(lags, spread[lags])

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
x = np.linspace(-120, 120, 600)
for t, colour in ((50, "#94a3b8"), (250, "#2563eb"), (1000, "#d97706")):
    centres, density = sampling.walker_histogram(walkers[:, t], n_bins=120, span=(-120, 120),
                                                 lattice=2 * STEP)
    left.step(centres, density, where="mid", color=colour, label=f"t = {t} tau")
    left.plot(x, transport.diffusion_greens_function(x, t * TAU, D_EXACT), color=colour, ls="--")
left.set_xlabel("x / l")
left.set_ylabel("P(x, t) l")
left.legend()
right.loglog(np.arange(1, 1001), spread[1:], color="k", label="measured")
right.loglog(lags, 2 * D_EXACT * lags * TAU, "--", color="#2563eb", label="2 D t")
right.set_xlabel("t / tau")
right.set_ylabel("<x^2> / l^2")
right.legend()
plt.tight_layout()
plt.show()
widths = {t: walkers[:, t].std() for t in (250, 1000)}
print(f"log-log slope of <x^2> from 10 to 1000 steps: {cloud_slope:.4f}")
print(f"width at 250 and 1000 steps: {widths[250]:.2f} l and {widths[1000]:.2f} l, "
      f"ratio {widths[1000] / widths[250]:.3f}")

Four times as long, twice as wide: the cloud's width grows as $\sqrt{t}$, the mean squared
displacement has slope one, and the Green's function — a prediction with no free parameter —
lies on the histogram at every time.

## Part 6 — $D$ three ways

Three measurements of the same $D$ that use three different features of the walkers: the growth
of the mean squared displacement (`d_from_msd`, dominated by the cloud's tails); a least-squares
fit of the histogram's shape to the Green's function at four times (`d_from_histogram`, dominated
by its middle); and the integral of the velocity autocorrelation (`d_from_vacf`, which uses no
positions at all). Each error bar comes from splitting the walkers into twenty independent batches.

**Your prediction:** which of the three will have the smallest error bar, and why?

*(write here before running the next cell)*

In [ ]:
velocities = transport.walker_velocities(walkers, TAU)
three = {
    "mean squared displacement": transport.d_from_msd(walkers, TAU),
    "Green's-function histogram fit": transport.d_from_histogram(walkers, TAU, lattice=2 * STEP),
    "velocity autocorrelation": transport.d_from_vacf(velocities, TAU),
}
for name, estimate in three.items():
    print(f"{name:32s} D = {estimate.value:.4f} +- {estimate.error:.4f}   "
          f"({(estimate.value - D_EXACT) / estimate.error:+.1f} errors from l^2/(2 tau) = 0.5)")
names = list(three)
consistent = {(a, b): three[a].consistent_with(three[b])
              for i, a in enumerate(names) for b in names[i + 1:]}
for (a, b), verdict in consistent.items():
    print(f"  {a} vs {b}: {'consistent' if verdict else 'NOT consistent'} within 2 combined errors")

vacf = transport.velocity_autocorrelation(velocities, 100)
running = transport.green_kubo_running(vacf, TAU)
print("C(k) / C(0) for k = 0 ... 5:", np.round(vacf[:6] / vacf[0], 4))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(np.arange(101), running, color="k", label="tau [C(0)/2 + C(1) + ... + C(K)]")
ax.axhline(D_EXACT, color="#2563eb", ls="--", label="l^2 / (2 tau)")
ax.set_xlabel("cut-off K (steps)")
ax.set_ylabel("running integral (l^2 / tau)")
ax.set_ylim(0, 0.7)
ax.legend()
plt.tight_layout()
plt.show()

Three different features of the data, one $D$. The velocity route is the most precise here, for
two reasons: every time origin of every walker contributes to it, while the mean squared
displacement uses each walker's start only; and every step is exactly $\pm\ell$, so $C(0)$ carries no
noise at all. For these walkers it mostly tests that the steps are independent — Part 7 is the
harder test. The velocity of an independent walker is forgotten after one step — $C(k)$ is
zero within noise for $k \geq 1$ — and the integral is $\tfrac12\tau C(0) = \ell^2/2\tau$.

## Part 7 — Walkers with memory

Now each walker repeats its last step with probability $q = 0.9$, and otherwise takes a fresh
random step, as a molecule keeps its velocity between collisions: module 03's persistent walk.
Steps are still $\pm\ell$ with equal probability, but no longer independent: steps $k$ apart are
correlated by $q^k$.

**Your prediction:** does $D = \ell^2/2\tau$ still hold? If not, by how much is it off?

*(write here before running the next cell)*

In [ ]:
rng = np.random.default_rng(1806)
Q = 0.9
memory = transport.walker_trajectories(5000, 1000, STEP, rng, persistence=Q)
memory_v = transport.walker_velocities(memory, TAU)
D_MEMORY = transport.persistent_walk_d(STEP, TAU, Q)
spread_q = transport.msd(memory)
short_slope = scaling_exponent(np.arange(1, 4), spread_q[1:4])
long_slope = scaling_exponent(np.arange(300, 1001, 50), spread_q[300:1001:50])
three_q = {
    "mean squared displacement": transport.d_from_msd(memory, TAU),
    "Green's-function histogram fit": transport.d_from_histogram(memory, TAU, lattice=2 * STEP),
    "velocity autocorrelation": transport.d_from_vacf(memory_v, TAU),
}
print(f"exact D = (l^2/(2 tau)) (1 + q)/(1 - q) = {D_MEMORY:.2f}; the formula l^2/(2 tau) "
      f"says {D_EXACT}")
for name, estimate in three_q.items():
    print(f"{name:32s} D = {estimate.value:.3f} +- {estimate.error:.3f}   "
          f"({(estimate.value - D_MEMORY) / estimate.error:+.1f} errors from the exact value)")
vacf_q = transport.velocity_autocorrelation(memory_v, 100)
memory_gap = float(np.max(np.abs(vacf_q[:21] / vacf_q[0] - Q ** np.arange(21))))
print(f"log-log slope of <x^2>: {short_slope:.3f} over steps 1-3, {long_slope:.3f} over 300-1000")
print(f"C(k)/C(0) against q^k for k <= 20: largest difference {memory_gap:.4f}")

# One run is one measurement. Repeat the whole experiment under six independent seeds.
repeats = {name: [] for name in three_q}
for seed in np.random.SeedSequence(1816).spawn(6):
    generator = np.random.default_rng(seed)
    trial = transport.walker_trajectories(2000, 1000, STEP, generator, persistence=Q)
    repeats["mean squared displacement"].append(transport.d_from_msd(trial, TAU).value)
    repeats["Green's-function histogram fit"].append(
        transport.d_from_histogram(trial, TAU, lattice=2 * STEP).value)
    repeats["velocity autocorrelation"].append(
        transport.d_from_vacf(transport.walker_velocities(trial, TAU), TAU).value)
print("\nsix independent repeats of 2000 walkers:")
for name, values in repeats.items():
    values = np.array(values)
    print(f"{name:32s} mean {values.mean():.3f} +- {values.std(ddof=1) / np.sqrt(values.size):.3f}")

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
t = np.arange(1, 1001)
left.loglog(t, spread_q[1:], color="k", label="measured")
left.loglog(t, (STEP / TAU) ** 2 * (t * TAU) ** 2, ":", color="#dc2626", label="<v^2> t^2")
left.loglog(t, 2 * D_MEMORY * t * TAU, "--", color="#2563eb", label="2 D t")
left.set_ylim(0.5, 3e4)
left.set_xlabel("t / tau")
left.set_ylabel("<x^2> / l^2")
left.legend()
right.plot(np.arange(41), vacf_q[:41] / vacf_q[0], "o", color="k", ms=4, label="measured")
right.plot(np.arange(41), Q ** np.arange(41), color="#2563eb", label="q^k")
right.set_xlabel("lag k (steps)")
right.set_ylabel("C(k) / C(0)")
right.legend()
plt.tight_layout()
plt.show()

The velocity memory decays as $q^k$, the mean squared displacement starts ballistic (slope two)
and turns diffusive (slope one) once the memory is spent, and $D$ is nineteen times
$\ell^2/2\tau$ — the factor $(1+q)/(1-q)$ that module 03 found in the variance of the persistent walk.
The independent-step formula fails because its hypothesis does; the Green–Kubo integral does not
fail, because it asks only how long the velocity is remembered. In the single run above, the three
estimates need not land within their error bars of one another and of the exact value — roughly one
measurement in twenty lands more than two error bars out — which is what the six repeats are for.

## Part 8 — Module 04's box, which cannot diffuse

Two hundred argon atoms in module 04's two-dimensional box, $1\ \mu\mathrm{m}$ on a side, with no
collisions between them, recorded step by step with `kinetics.simulate` unchanged.

**Your prediction:** what does the mean squared displacement of the box gas do at long times?

*(write here before running the next cell)*

In [ ]:
rng = np.random.default_rng(1807)
SIDE = 1e-6
box_state = kinetics.initialise_gas(200, (SIDE, SIDE), 300.0, M_AR, rng)
box_dt = kinetics.max_stable_dt(box_state)
box_x, box_v = transport.box_gas_trace(box_state, box_dt, 2000)
box_spread = transport.msd(box_x[:, :, 0].T)
box_vacf = transport.velocity_autocorrelation(box_v[:, :, 0].T, 1500)
box_running = transport.green_kubo_running(box_vacf, box_dt)
box_scale = kinetics.mean_speed(300.0, M_AR) * SIDE
saturation = float(box_spread[-400:].mean() / (SIDE**2 / 6))
crossings = 2000 * box_dt * kinetics.mean_speed(300.0, M_AR) / SIDE
print(f"run: {2000 * box_dt * 1e9:.0f} ns, about {crossings:.0f} box crossings")
print(f"late <x^2> / (L^2/6) = {saturation:.3f}")
print(f"C(s)/C(0): lowest {box_vacf.min() / box_vacf[0]:.2f}, "
      f"at s = {box_vacf.argmin() * box_dt * 1e9:.2f} ns")
print(f"running integral of C: largest {box_running.max() / box_scale:.3f} vbar L, "
      f"final {box_running[-1] / box_scale:+.3f} vbar L")

times_ns = np.arange(box_spread.size) * box_dt * 1e9
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(times_ns, box_spread / SIDE**2, color="k")
left.axhline(1 / 6, color="#2563eb", ls="--", label="L^2 / 6")
left.set_xlabel("t (ns)")
left.set_ylabel("<x^2> / L^2")
left.legend()
lag_ns = np.arange(box_vacf.size) * box_dt * 1e9
right.plot(lag_ns, box_vacf / box_vacf[0], color="k", label="C(s) / C(0)")
right.plot(lag_ns, box_running / box_scale, color="#d97706", label="running integral / (vbar L)")
right.axhline(0.0, color="grey", lw=1)
right.set_xlabel("s (ns)")
right.legend()
plt.tight_layout()
plt.show()

The box gas flies, hits a wall, and stops spreading at $L^2/6$ — the mean squared distance between
two independent positions in the box. Its velocity is reversed by the walls instead of being
forgotten: $C(s)$ swings negative and its integral returns to zero. With no collisions between
particles there is no diffusion, which is module 04's failure mode seen from inside.

## Part 9 — Collisions in a real gas

The molecular size comes from module 02's van der Waals $b$, which `gases.vdw_constants_from_critical`
obtains from the critical point; the critical constants are the NIST Chemistry WebBook's. Then the
mean free path and collision rate of nitrogen at $0\,{}^\circ\mathrm{C}$ and one atmosphere, and what
they mean for a walker given physical units: one step a free flight, one step time the time
between collisions.

**Your prediction:** how many molecular diameters does a nitrogen molecule fly between collisions at
room conditions — about $3$, about $30$ or about $300$?

*(write here before running the next cell)*

In [ ]:
# Critical temperature [K], critical pressure [Pa] and molar mass [kg/mol] (NIST Chemistry WebBook).
CRITICAL = {"He": (5.1953, 0.22832e6, 4.002602e-3),
            "N2": (126.192, 3.3958e6, 28.0134e-3),
            "Ar": (150.687, 4.863e6, 39.948e-3)}
DIAMETER = {name: transport.diameter_from_vdw_b(gases.vdw_constants_from_critical(tc, pc)[1])
            for name, (tc, pc, _) in CRITICAL.items()}
for name, d in DIAMETER.items():
    print(f"{name}: van der Waals diameter {d * 1e9:.3f} nm")

n_stp = 101325.0 / (K_B * 273.15)
n2 = transport.kinetic_estimates(n_stp, 273.15, CRITICAL["N2"][2] / N_A,
                                 transport.hard_sphere_cross_section(DIAMETER["N2"]), 2.5 * K_B)
spacing = n_stp ** (-1 / 3)
print(f"\nnitrogen at 0 C and 1 atm: n = {n_stp:.3e} m^-3, mean spacing {spacing * 1e9:.2f} nm")
print(f"mean free path {n2.mean_free_path * 1e9:.0f} nm = {n2.mean_free_path / DIAMETER['N2']:.0f} "
      f"diameters = {n2.mean_free_path / spacing:.0f} spacings")
print(f"mean speed {n2.mean_speed:.0f} m/s, collision rate {n2.collision_rate:.2e} per second")
walker_d = n2.mean_free_path**2 * n2.collision_rate / 2
print(f"\nas a walker with l = lambda and tau = 1/nu: D = l^2/(2 tau) = {walker_d:.2e} m^2/s (1-D)")
print(f"as a 3-D random flight: D = vbar lambda / 3 = {n2.diffusion:.2e} m^2/s")

About $270$ diameters: a dilute gas, as module 04 assumed, with collisions rare on the scale of the
molecules and frequent — five billion a second — on any scale a person can see. A one-dimensional
walker whose every step is a full mean free path overestimates $D$ by $\tfrac32$: in three
dimensions only a third of a flight's mean square lies along any one axis, and the exponential spread
of flight lengths doubles that mean square.

## Part 10 — Viscosity and conductivity, against measurement

The data file holds the viscosity and thermal conductivity of helium, nitrogen and argon at
$300\ \mathrm{K}$ from $10^{-3}$ to $10\ \mathrm{MPa}$, from the NIST Chemistry WebBook's reference
correlations — equations fitted to many measurements, with stated uncertainties of $0.5\%$
(viscosity) and $2\%$ (conductivity) for nitrogen and argon, $10\%$ and $5\%$ for helium. They are
compared with the kinetic estimates of step 8, using the van der Waals diameters of Part 9, and
with measured self-diffusion coefficients. The ratio $\kappa/(\eta c_v)$, with $c_v$ per kilogram,
is $1$ in the crude estimates whatever the molecular size.

**Your prediction:** by what fraction does the viscosity of nitrogen change between $0.1$ and
$1\ \mathrm{MPa}$ — a tenfold change of density?

*(write here before running the next cell)*

In [ ]:
nist = read_table("18-gas-transport.csv")
N_300 = 1e5 / (K_B * 300.0)   # number density at 0.1 MPa and 300 K
comparison = {}
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
print("300 K, 0.1 MPa     eta: estimate / NIST    kappa: estimate / NIST    kappa / (eta c_v)    "
      "eta, 0.1 -> 1 MPa")
for gas, colour in (("He", "#d97706"), ("N2", "#2563eb"), ("Ar", "#111827")):
    rows = nist["gas"] == gas
    pressure = nist["pressure_MPa"][rows] * 1e6
    eta, kappa = nist["viscosity_uPa_s"][rows] * 1e-6, nist["thermal_conductivity_W_mK"][rows]
    cv_mass = nist["cv_J_gK"][rows] * 1e3
    i_low, i_high = np.argmin(np.abs(pressure - 1e5)), np.argmin(np.abs(pressure - 1e6))
    c_v = (2.5 if gas == "N2" else 1.5) * K_B
    est = transport.kinetic_estimates(N_300, 300.0, CRITICAL[gas][2] / N_A,
                                      transport.hard_sphere_cross_section(DIAMETER[gas]), c_v)
    comparison[gas] = {
        "eta": est.viscosity / eta[i_low],
        "kappa": est.thermal_conductivity / kappa[i_low],
        "eucken": kappa[i_low] / (eta[i_low] * cv_mass[i_low]),
        "flat": eta[i_high] / eta[i_low] - 1,
    }
    c = comparison[gas]
    print(f"   {gas:3s}               {c['eta']:6.3f}                  {c['kappa']:6.3f}"
          f"                 {c['eucken']:6.3f}            {100 * c['flat']:+.2f}%")
    left.semilogx(pressure, eta * 1e6, color=colour, label=f"{gas} (NIST)")
    left.axhline(est.viscosity * 1e6, color=colour, ls="--", lw=1)
    right.semilogx(pressure, kappa * 1e3, color=colour, label=f"{gas} (NIST)")
    right.axhline(est.thermal_conductivity * 1e3, color=colour, ls="--", lw=1)
for ax in (left, right):
    ax.axvspan(1e5, 1e6, color="#e5e7eb", zorder=0)
    ax.set_xlabel("pressure (Pa)")
    ax.legend()
left.set_ylabel("viscosity (uPa s); dashed: kinetic estimate")
right.set_ylabel("thermal conductivity (mW/(m K))")
right.set_yscale("log")
plt.tight_layout()
plt.show()

# Measured self-diffusion at 273.2 K and one atmosphere, as tabulated by Hirschfelder, Curtiss
# and Bird, Molecular Theory of Gases and Liquids (1954), Table 8.4-13, p. 581 [m^2/s]:
# argon 0.156e-4 +- 0.002e-4 (Winn 1950) and 0.158e-4 +- 0.002e-4 (Hutchinson 1949);
# nitrogen 0.185e-4 +- 0.006e-4 (Winn 1950) and 0.172e-4 +- 0.002e-4 (Winter 1951).
SELF_DIFFUSION = {"Ar": (0.156e-4, 0.158e-4), "N2": (0.185e-4, 0.172e-4)}
for gas, measured in SELF_DIFFUSION.items():
    est = transport.kinetic_estimates(101325.0 / (K_B * 273.2), 273.2, CRITICAL[gas][2] / N_A,
                                      transport.hard_sphere_cross_section(DIAMETER[gas]))
    print(f"self-diffusion of {gas} at 273 K, 1 atm: estimate {est.diffusion:.3e} m^2/s, measured "
          f"{measured[0]:.3e} and {measured[1]:.3e} "
          f"(ratio {est.diffusion / np.mean(measured):.2f})")

Over the shaded tenfold change of density the viscosity moves by less than one per cent, as
Maxwell predicted; it rises only at megapascals, where the gas is no longer dilute. (Below
$0.1\ \mathrm{MPa}$ the curves are flat by construction: that is the dilute-gas form the
correlations assume, and no test of it.) The estimates themselves are honest to a factor of order
one, and their good agreement for nitrogen and argon is partly luck: the careful theory's
coefficient is about $1.5$ times the crude $\tfrac13$, and the van der Waals diameter happens to be
smaller than a collision diameter by about the amount that compensates; for helium the two errors
add. The ratio $\kappa/(\eta c_v)$ is a cleaner test, since the size cancels: the crude estimates
say $1$, the monatomic gases say $2.51$ — the careful theory's $\tfrac52$ — because the fastest
molecules carry the most energy and fly the furthest.

## Part 11 — Perrin's atoms

The data file is a table from Perrin's 1909 memoir: the horizontal projections of $30$-second
displacements of gamboge granules of radius $0.212\ \mu\mathrm{m}$, counted by M. Chaudesaigues in
bins of $1.7\ \mu\mathrm{m}$, for two series of $50$ granules. The first series was in water at
$17\,{}^\circ\mathrm{C}$, viscosity $0.011$ poise ($1.1 \times 10^{-3}\ \mathrm{Pa\,s}$); the second
series' temperature is not given. The histogram is fitted to a Gaussian in $|x|$ — the same
estimator as Part 6's, applied to a table — with a bootstrap error bar, then turned into $D$ and,
through the Stokes–Einstein relation, into Avogadro's number.

**Your prediction:** to within what fraction will two hundred displacements of a resin grain
measure Avogadro's number?

*(write here before running the next cell)*

In [ ]:
perrin = read_table("18-perrin-projections.csv")
edges = np.append(perrin["low_um"], perrin["high_um"][-1]) * 1e-6
rng = np.random.default_rng(1811)
RADIUS, ETA_WATER, T_WATER, INTERVAL = 0.212e-6, 1.1e-3, 290.15, 30.0
xi2 = {series: transport.binned_variance_estimate(edges, perrin[f"{series}_found"], rng)
       for series in ("first", "second")}
for series, estimate in xi2.items():
    print(f"{series} series: {int(perrin[f'{series}_found'].sum())} projections, "
          f"fitted <x^2> = {estimate.value * 1e12:.1f} +- {estimate.error * 1e12:.1f} um^2")
d_perrin = xi2["first"].value / (2 * INTERVAL)
d_perrin_error = xi2["first"].error / (2 * INTERVAL)
n_a = transport.avogadro_from_diffusion(d_perrin, T_WATER, ETA_WATER, RADIUS)
n_a_error = n_a * d_perrin_error / d_perrin
stokes_einstein = transport.einstein_diffusion(T_WATER,
                                               transport.stokes_friction(ETA_WATER, RADIUS))
print(f"first series: D = {d_perrin:.2e} +- {d_perrin_error:.1e} m^2/s; "
      f"Stokes-Einstein with the defined k_B predicts {stokes_einstein:.2e}")
print(f"Avogadro's number: {n_a:.2e} +- {n_a_error:.1e} per mole (defined value {N_A:.6e})")
counts, centres = perrin["first_found"], 0.5 * (edges[:-1] + edges[1:])
readings = {"second moment of the bin centres": np.sum(counts * centres**2) / counts.sum(),
            "(pi/2) <|x|>^2": np.pi / 2 * (np.sum(counts * centres) / counts.sum()) ** 2}
for name, value in readings.items():
    other = transport.avogadro_from_diffusion(value / (2 * INTERVAL), T_WATER, ETA_WATER, RADIUS)
    print(f"  the same table read as its {name}: <x^2> = {value * 1e12:.1f} um^2 "
          f"-> N_A = {other:.2e}")

# Perrin's own <x^2> for the first series at four intervals (p. 77, in square micrometres),
# from which he computed N = 66, 59, 78 and 89 x 10^22.
PERRIN_XI2 = {30: 50.2, 60: 113.5, 90: 128.0, 120: 144.0}
for interval, value in PERRIN_XI2.items():
    n_from_table = transport.avogadro_from_diffusion(value * 1e-12 / (2 * interval), T_WATER,
                                                     ETA_WATER, RADIUS)
    print(f"  {interval:3d} s: <x^2> = {value} um^2 -> N_A = {n_from_table:.2e}")

# Perrin's summary of N from seven emulsions (Atoms, 1916, p. 123): radius [um], N / 10^22.
PERRIN_EMULSIONS = [(0.50, 80), (0.212, 69.5), (0.212, 55), (0.52, 72.5), (5.50, 78),
                    (0.385, 64), (0.367, 68.8)]
values = np.array([n for _, n in PERRIN_EMULSIONS]) * 1e22
print(f"seven emulsions, radii 0.21 to 5.5 um: N = {values.mean():.2e}, spread "
      f"{values.std(ddof=1) / values.mean():.0%}")

# The fitted Gaussian's probability for each bin of |x|: a difference of erf at the edges.
model = np.diff([math.erf(e / math.sqrt(2 * xi2["first"].value)) for e in edges])
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(centres * 1e6, perrin["first_found"], width=1.6, color="#93c5fd",
       label="found (first series)")
ax.plot(centres * 1e6, model / model.sum() * perrin["first_found"].sum(), "o-", color="#2563eb",
        label="fitted Gaussian in |x|")
ax.set_xlabel("|x| (micrometres)")
ax.set_ylabel("displacements")
ax.legend()
plt.tight_layout()
plt.show()

Two hundred and twenty-one numbers written down by hand in 1908, and the same estimator as for
the walkers, give Avogadro's number consistent with the defined value. The $\pm$ is counting noise
only, and the choice of estimator matters as much: the table's own second moment, and its mean
$|x|$, give $42$ and $45\ \mu\mathrm{m}^2$ instead of the fit's $50$, because the fit leans on one
over-full bin. Perrin's per-interval values scatter more still, which is honest counting noise in
fifty granules followed for two minutes. Seven emulsions, with radii from $0.2$ to $5.5\ \mu\mathrm{m}$
and masses differing fifteen-thousand-fold, all gave $N$ between $5.5$ and $8 \times 10^{23}$: a
number that does not care what it is measured with is a property of the world.

## Part 12 — A diffusion experiment of your own

Diffusion in a jar of water is almost impossible to see: small temperature differences drive
convection that carries dye centimetres in minutes. Gelatin stops convection and lets small molecules
diffuse almost as in water. Make up a packet of clear gelatin in a tall glass and let it set
overnight; pour a layer of strongly coloured food dye on top; keep the glass at constant temperature
and photograph it against a ruler every hour or two for a day. At each time, measure how deep the
colour has reached a shade you choose — the same shade every time — and enter the times and depths
below. A diffusion front advances as $\sqrt{t}$: the log-log slope of depth against time should be
$\tfrac12$. The shade you chose fixes an unknown multiple of $\sqrt{Dt}$, so $D$ itself comes out only
to within that factor, but the exponent does not depend on it.

In [ ]:
my_hours = []        # for example [1, 2, 4, 8, 16, 24]
my_depths_mm = []    # the depth of your chosen shade at each of those times
if len(my_hours) >= 3:
    hours, depths = np.array(my_hours, dtype=float), np.array(my_depths_mm, dtype=float)
    print(f"log-log slope of depth against time: {scaling_exponent(hours, depths):.2f} "
          f"(diffusion 0.5, steady speed 1)")
    spread = np.sum(hours * 3600 * (depths * 1e-3) ** 2) / np.sum((hours * 3600) ** 2)
    print(f"depth^2 / t = {spread:.2e} m^2/s; for a quarter-strength shade, D = that / 0.910 "
          f"= {spread / 0.910:.1e} m^2/s")
else:
    print("No measurements yet: fill in my_hours and my_depths_mm, then run this cell again.")

### Measurement

Quote three results, each with its uncertainty and the assumptions it rests on:

- $D$ for the independent walkers of Part 6, three ways, with your verdict on whether they are
  consistent — and what an inconsistency would have meant.
- The heat capacity of $100$ argon atoms from the noise of Part 2, against equipartition.
- Avogadro's number from Perrin's data in Part 11, against the defined value
  $6.02214076 \times 10^{23}\ \mathrm{mol^{-1}}$.

## Part 13 — The automated checks

In [ ]:
# 1. The noise is a calorimeter, and only on a bath.
assert abs(argon.value / (100 * K_B) - 1.5) < 4 * argon.error / (100 * K_B)
assert abs(four[True].value / (4 * K_B) - 1.125) < 4 * four[True].error / (4 * K_B)
assert fixed.value < 1e-12 * 100 * K_B
assert all(abs(noise.value - exact) < 4 * noise.error for noise, _, _, exact in two_level.values())

# 2. A longer run measures the fluctuation better; it does not make it smaller.
assert abs(longer[100_000][0] / exact_sigma - 1) < 0.02
assert longer[100_000][1] < longer[100][1] / 25

# 3. sigma_E / <E> ~ N^(-1/2).
assert abs(exponent + 0.5) < 0.05

# 4. Diffusion: slope one, and three independent measurements of l^2/(2 tau).
assert abs(cloud_slope - 1.0) < 0.02
assert all(abs(e.value - D_EXACT) < 4 * e.error for e in three.values())

# 5. Memory: ballistic then diffusive; D inflated by (1 + q)/(1 - q), as Green-Kubo says.
assert short_slope > 1.8 and abs(long_slope - 1.0) < 0.1 and memory_gap < 0.02
assert all(abs(e.value - D_MEMORY) < 4 * e.error for e in three_q.values())
assert all(abs(np.mean(v) - D_MEMORY) / D_MEMORY < 0.05 for v in repeats.values())

# 6. Module 04's box saturates at L^2/6 and never diffuses.
assert abs(saturation - 1) < 0.1 and abs(box_running[-1]) < 0.05 * box_scale

# 7. Nitrogen at room conditions is dilute.
assert 50e-9 < n2.mean_free_path < 150e-9 and n2.mean_free_path / spacing > 10

# 8. Maxwell: a tenfold change of density moves a dilute gas's viscosity by under 1%; the
#    estimates are honest to a factor of order one; kappa / (eta c_v) is 5/2 for monatomic gases.
assert all(abs(c["flat"]) < 0.01 for c in comparison.values())
assert all(0.3 < c["eta"] < 3 and 0.1 < c["kappa"] < 3 for c in comparison.values())
assert abs(comparison["Ar"]["eucken"] - 2.5) < 0.05 and abs(comparison["He"]["eucken"] - 2.5) < 0.05

# 9. Perrin's granules weigh the atom to within their counting noise.
assert abs(xi2["first"].value * 1e12 - PERRIN_XI2[30]) < 2 * xi2["first"].error * 1e12
assert abs(n_a - N_A) < 3 * n_a_error
print("all checks passed")

## Part 14 — Explore it yourself

Four explorers; set the controls and press **Run Interact**.

**The walker cloud.** Release up to ten thousand walkers and look at their histogram at the time you
choose, with the Green's function for the walkers' long-time $D$ over it, and their mean squared
displacement. Things to try: $250$ and then $1000$ steps; the logarithmic axes; then a memory of
$0.95$, and an early snapshot.

In [ ]:
import ipywidgets as widgets


def cloud(n_walkers=4000, snapshot=250, memory=0.0, log_axes=True, seed=1):
    generator = np.random.default_rng(seed)
    x_cloud = transport.walker_trajectories(n_walkers, 1000, STEP, generator, persistence=memory)
    d_long = transport.persistent_walk_d(STEP, TAU, memory)
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
    width = 4 * np.sqrt(2 * d_long * snapshot * TAU) + 10
    centres, density = sampling.walker_histogram(x_cloud[:, snapshot], n_bins=60,
                                                 span=(-width, width), lattice=2 * STEP)
    left.bar(centres, density, width=centres[1] - centres[0], color="#93c5fd")
    grid = np.linspace(-width, width, 400)
    left.plot(grid, transport.diffusion_greens_function(grid, snapshot * TAU, d_long),
              color="#2563eb")
    left.set_xlabel("x / l")
    left.set_ylabel("P(x, t) l")
    t = np.arange(1, 1001)
    measured = transport.msd(x_cloud)[1:]
    if log_axes:
        right.loglog(t, measured, color="k")
        right.loglog(t, 2 * d_long * t * TAU, "--", color="#2563eb")
    else:
        right.plot(t, measured, color="k")
        right.plot(t, 2 * d_long * t * TAU, "--", color="#2563eb")
    right.axvline(snapshot, color="#d97706", lw=1)
    right.set_xlabel("t / tau")
    right.set_ylabel("<x^2> / l^2")
    plt.tight_layout()
    plt.show()
    estimate = transport.d_from_msd(x_cloud, TAU)
    print(f"D from the mean squared displacement: {estimate.value:.3f} +- {estimate.error:.3f}; "
          f"exact {d_long:.3f}")


widgets.interact_manual(
    cloud,
    n_walkers=widgets.IntSlider(value=4000, min=100, max=10_000, step=100),
    snapshot=widgets.IntSlider(value=250, min=5, max=1000, step=5),
    memory=widgets.FloatSlider(value=0.0, min=0.0, max=0.98, step=0.01),
    log_axes=widgets.Checkbox(value=True, description="log axes"),
    seed=widgets.IntText(value=1),
)

**The fluctuation explorer.** The energies of $N$ two-level systems drawn from the canonical
distribution, as a histogram of $E/N$, with the relative fluctuation against the identity's
prediction. Things to try: $N = 16$ and then $N = 4096$; then a temperature well below the gap.

In [ ]:
def fluctuations(n_systems=64, kt_over_gap=1.0, seed=1):
    generator = np.random.default_rng(seed)
    t = kt_over_gap * GAP / K_B
    energies = transport.two_level_energy_samples(n_systems, GAP, t, 4000, generator)
    c = transport.energy_fluctuation_cv(energies, t)
    exact = n_systems * partition.two_level_heat_capacity(GAP, t)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(energies / (n_systems * GAP), bins=40, color="#93c5fd")
    ax.set_xlabel("E / (N gap)")
    ax.set_ylabel("samples")
    plt.tight_layout()
    plt.show()
    predicted = np.sqrt(K_B * t**2 * exact) / (n_systems * partition.two_level_energy(GAP, t))
    print(f"sigma_E / <E> = {energies.std(ddof=1) / energies.mean():.4f}; "
          f"sqrt(k_B T^2 C_V) / U = {predicted:.4f}")
    print(f"C_V from the noise / (N k_B) = {c.value / (n_systems * K_B):.4f} +- "
          f"{c.error / (n_systems * K_B):.4f}; module 12: {exact / (n_systems * K_B):.4f}")


widgets.interact_manual(
    fluctuations,
    n_systems=widgets.SelectionSlider(options=[int(n) for n in 2 ** np.arange(4, 13)], value=64),
    kt_over_gap=widgets.FloatLogSlider(value=1.0, base=10, min=-1.0, max=1.0, step=0.05),
    seed=widgets.IntText(value=1),
)

**The transport calculator.** The mean free path, collision rate and transport trio of helium,
nitrogen or argon at the temperature and pressure you choose. Things to try: nitrogen from $10^5$
down to $10^3\ \mathrm{Pa}$; argon against helium at the same conditions; a vacuum of
$10^{-4}\ \mathrm{Pa}$, and the Knudsen number for a $10\ \mathrm{cm}$ chamber.

In [ ]:
def transport_calculator(gas="N2", temperature=300.0, log10_pressure=5.0):
    tc, pc, molar_mass = CRITICAL[gas]
    pressure = 10.0**log10_pressure
    n = pressure / (K_B * temperature)
    c_v = (2.5 if gas == "N2" else 1.5) * K_B
    est = transport.kinetic_estimates(n, temperature, molar_mass / N_A,
                                      transport.hard_sphere_cross_section(DIAMETER[gas]), c_v)
    print(f"{gas} at {temperature:.0f} K and {pressure:.3g} Pa: n = {n:.3e} m^-3")
    print(f"  mean free path {est.mean_free_path:.3e} m; Knudsen number in a 10 cm chamber "
          f"{transport.knudsen_number(est.mean_free_path, 0.1):.2e}")
    print(f"  collision rate {est.collision_rate:.3e} per second; "
          f"mean speed {est.mean_speed:.0f} m/s")
    print(f"  D ~ {est.diffusion:.3e} m^2/s, eta ~ {est.viscosity * 1e6:.2f} uPa s, "
          f"kappa ~ {est.thermal_conductivity * 1e3:.2f} mW/(m K)")


widgets.interact_manual(
    transport_calculator,
    gas=widgets.Dropdown(options=["He", "N2", "Ar"], value="N2"),
    temperature=widgets.FloatSlider(value=300.0, min=50.0, max=1500.0, step=10.0),
    log10_pressure=widgets.FloatSlider(value=5.0, min=-4.0, max=7.0, step=0.25),
)

**The fronts race.** A dye layer diffusing into clear water against a front moving at a steady
speed, both shown at the time you choose. Things to try: the time at which the steady front
overtakes; then multiply the time by four and compare the distances.

In [ ]:
def fronts(d_dye=4e-10, steady_speed_mm_per_hour=1.0, hours=4.0):
    t = hours * 3600.0
    height = np.linspace(-5e-3, 25e-3, 600)
    profile = transport.step_profile(height, t, d_dye)
    # The quarter-strength contour of (1/2) erfc(x / 2 sqrt(D t)) sits at 0.954 sqrt(D t).
    diffusive = 0.9539 * np.sqrt(d_dye * t)
    steady = steady_speed_mm_per_hour * 1e-3 * hours
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.fill_between(height * 1e3, profile, color="#93c5fd")
    ax.axvline(diffusive * 1e3, color="#2563eb", ls="--", label="diffusive front (c = c0/4)")
    ax.axvline(steady * 1e3, color="#dc2626", label="steady front")
    ax.set_xlabel("height above the layer (mm)")
    ax.set_ylabel("c / c0")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"after {hours:g} h: diffusive front {diffusive * 1e3:.2f} mm, "
          f"steady front {steady * 1e3:.2f} mm")


widgets.interact_manual(
    fronts,
    d_dye=widgets.FloatLogSlider(value=4e-10, base=10, min=-10.0, max=-8.0, step=0.05),
    steady_speed_mm_per_hour=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1),
    hours=widgets.FloatSlider(value=4.0, min=0.25, max=48.0, step=0.25),
)

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "18-fluctuations-transport.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. Part 3 ran a thousand times longer and the fluctuation did not shrink. What did shrink, and
   why is that the difference between a property and an error?
3. In Part 7 a single run's three estimates of $D$ need not agree within their error bars. How
   would you decide whether a disagreement is bad luck or a real problem with a method?
4. Module 04's gas cannot diffuse. What would you have to add to its model to make it diffuse, and
   which line of its model specification would change?

**Your answers:**

1.
2.
3.
4.